<a href="https://colab.research.google.com/github/TissianyDelmiro/rag-analise-contratos/blob/main/rag_analise_contratos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-groq langchain-huggingface langchain-community langchain-text-splitters chromadb sentence-transformers tiktoken

# Configuração da API Key

In [ ]:
import os

In [ ]:
os.environ["GROQ_API_KEY"] = "sua_chave_aqui"  # substitua pela sua chave da Groq

# O documento do cliente

In [ ]:
documento = """
CONTRATO DE FORNECIMENTO — Xperiun S.A.

Cláusula 1 — Reajuste
O contrato prevê reajuste anual pelo IPCA.
O índice é calculado em dezembro e aplicado em janeiro do ano seguinte.
Caso o IPCA supere 10%, as partes se reúnem para renegociar os valores.
O reajuste é comunicado com 30 dias de antecedência.

Cláusula 2 — Vigência
A vigência é de 24 meses a partir de janeiro de 2024.
A renovação é automática por mais 12 meses, salvo notificação contrária
com 60 dias de antecedência do término.

Cláusula 3 — Entrega
O fornecedor se compromete a entregar os materiais em até 15 dias úteis
após a emissão do pedido de compra.
Atrasos superiores a 5 dias úteis geram multa de 0,5% ao dia sobre
o valor do pedido, limitada a 10% do valor total.

Cláusula 4 — Pagamento
O pagamento é realizado em 30 dias após a entrega e aceite da nota fiscal.
Pagamentos em atraso incorrem em juros de 1% ao mês mais correção pelo IGPM.

Cláusula 5 — Rescisão
Qualquer das partes pode rescindir o contrato com aviso prévio de 90 dias.
Rescisão imotivada pelo contratante gera multa de 20% sobre o valor
restante do contrato.
"""

print(f"   Documento carregado")
print(f"   Total de caracteres: {len(documento)}") # len calcula o comprimento ou o número de elementos nesses pedaços
print(f"   Total de palavras aproximado: {len(documento.split())}") # split divide uma string em pedaços com base em um separador

   Documento carregado
   Total de caracteres: 1104
   Total de palavras aproximado: 197


# Chunking com overlap

In [ ]:
# Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap  = 50,
    length_function = len
)

chunks = splitter.create_documents([documento])

print(f"   Chunking concluído")
print(f"   Documento original: {len(documento)} caracteres")
print(f"   Total de chunks gerados: {len(chunks)}")
print()

   Chunking concluído
   Documento original: 1104 caracteres
   Total de chunks gerados: 6



In [ ]:
for i, chunk in enumerate(chunks):
    print(f"--- CHUNK {1+1} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content)
    print()

--- CHUNK 2 (39 chars) ---
CONTRATO DE FORNECIMENTO — Xperiun S.A.

--- CHUNK 2 (261 chars) ---
Cláusula 1 — Reajuste
O contrato prevê reajuste anual pelo IPCA.
O índice é calculado em dezembro e aplicado em janeiro do ano seguinte.
Caso o IPCA supere 10%, as partes se reúnem para renegociar os valores.
O reajuste é comunicado com 30 dias de antecedência.

--- CHUNK 2 (187 chars) ---
Cláusula 2 — Vigência
A vigência é de 24 meses a partir de janeiro de 2024.
A renovação é automática por mais 12 meses, salvo notificação contrária
com 60 dias de antecedência do término.

--- CHUNK 2 (245 chars) ---
Cláusula 3 — Entrega
O fornecedor se compromete a entregar os materiais em até 15 dias úteis
após a emissão do pedido de compra.
Atrasos superiores a 5 dias úteis geram multa de 0,5% ao dia sobre
o valor do pedido, limitada a 10% do valor total.

--- CHUNK 2 (174 chars) ---
Cláusula 4 — Pagamento
O pagamento é realizado em 30 dias após a entrega e aceite da nota fiscal.
Pagamentos em atraso in

# Modelo de Embedding

#### Texto → tokens (IDs sem significado) → embedding (vetor com significado)

In [ ]:
# Modelo de Embedding
# Configurar e testar o modelo que vai gerar os vetores
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"  # 768 dims
)

frase_teste = "reajuste anual pelo IPCA"
vetor_teste = embedding_model.embed_query(frase_teste)

print(f"   Modelo de embedding configurado")
print(f"   Frase: '{frase_teste}'")
print(f"   Dimensões do vetor: {len(vetor_teste)}")  # vai dar 768

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


   Modelo de embedding configurado
   Frase: 'reajuste anual pelo IPCA'
   Dimensões do vetor: 768


# Popular o Banco Vetorial

In [ ]:
# Popular o Banco Vetorial
# Cada chunk vira um vetor e é salvo no banco

from langchain_community.vectorstores import Chroma

# from_documents faz tudo de uma vez:
# 1. pega cada chunk
# 2. manda pro modelo de embedding
# 3. recebe o vetor
# 4. salva no banco
banco = Chroma.from_documents(
    documents=chunks,         # os pedaços do documento
    embedding=embedding_model,  # o modelo que gera os vetores
)

print(f"   Banco vetorial populado")
print(f"   Total de documentos no banco: {banco._collection.count()}")

   Banco vetorial populado
   Total de documentos no banco: 6


In [ ]:
primeiros_5_numeros = []

for n in vetor_teste[:5]:
    numero_arredondado = round(n, 4)
    primeiros_5_numeros.append(numero_arredondado)

print(f"   Primeiros 5 números do vetor: {primeiros_5_numeros}")
print()



   Primeiros 5 números do vetor: [0.0107, -0.0314, -0.0294, 0.0004, 0.016]



In [ ]:
# EXPLORANDO O BANCO VETORIAL

todos = banco._collection.get(
    include=["documents", "embeddings"]
)

print("Chunks armazenados:")

for i, doc in enumerate(todos["documents"]):
    print(f"\n--- Chunk {i+1} ---")
    print(doc)

print()
print("Embedding do primeiro chunk (primeiros 10 números):")

Chunks armazenados:

--- Chunk 1 ---
CONTRATO DE FORNECIMENTO — Xperiun S.A.

--- Chunk 2 ---
Cláusula 1 — Reajuste
O contrato prevê reajuste anual pelo IPCA.
O índice é calculado em dezembro e aplicado em janeiro do ano seguinte.
Caso o IPCA supere 10%, as partes se reúnem para renegociar os valores.
O reajuste é comunicado com 30 dias de antecedência.

--- Chunk 3 ---
Cláusula 2 — Vigência
A vigência é de 24 meses a partir de janeiro de 2024.
A renovação é automática por mais 12 meses, salvo notificação contrária
com 60 dias de antecedência do término.

--- Chunk 4 ---
Cláusula 3 — Entrega
O fornecedor se compromete a entregar os materiais em até 15 dias úteis
após a emissão do pedido de compra.
Atrasos superiores a 5 dias úteis geram multa de 0,5% ao dia sobre
o valor do pedido, limitada a 10% do valor total.

--- Chunk 5 ---
Cláusula 4 — Pagamento
O pagamento é realizado em 30 dias após a entrega e aceite da nota fiscal.
Pagamentos em atraso incorrem em juros de 1% ao mês mais corr

# Testando a Busca Vetorial

In [ ]:
# Vamos ver o que o banco retorna ANTES de conectar o LLM

pergunta = "Qual é a cláusula de reajuste?"

# similarity_search:
# 1. embeda a pergunta (mesmo modelo de embedding)
# 2. calcula distância para todos os chunks no banco
# 3. retorna os k mais próximos
chunks_encontrados = banco.similarity_search(
    query=pergunta,
    k=3  # retorna os 3 vizinhos mais próximos
)

print(f"Pergunta: '{pergunta}'")
print(f"Chunks encontrados: {len(chunks_encontrados)}")
print()

# Esses são os trechos que vão pro prompt do LLM
for i, chunk in enumerate(chunks_encontrados):
    print(f"--- CHUNK ENCONTRADO {i+1} ---")
    print(chunk.page_content)
    print()

# O LLM ainda não foi chamado — isso é só a busca

Pergunta: 'Qual é a cláusula de reajuste?'
Chunks encontrados: 3

--- CHUNK ENCONTRADO 1 ---
Cláusula 1 — Reajuste
O contrato prevê reajuste anual pelo IPCA.
O índice é calculado em dezembro e aplicado em janeiro do ano seguinte.
Caso o IPCA supere 10%, as partes se reúnem para renegociar os valores.
O reajuste é comunicado com 30 dias de antecedência.

--- CHUNK ENCONTRADO 2 ---
Cláusula 5 — Rescisão
Qualquer das partes pode rescindir o contrato com aviso prévio de 90 dias.
Rescisão imotivada pelo contratante gera multa de 20% sobre o valor
restante do contrato.

--- CHUNK ENCONTRADO 3 ---
Cláusula 2 — Vigência
A vigência é de 24 meses a partir de janeiro de 2024.
A renovação é automática por mais 12 meses, salvo notificação contrária
com 60 dias de antecedência do término.



# Configura o LLM Principal

In [ ]:
# LLM Principal

from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0, # determinístico — sem variação na resposta
)

# Teste rápido — só pra confirmar que funciona
resposta_teste = llm.invoke("Responda em uma palavra: o céu é qual cor?")

print(f" LLM configurado")
print(f" Teste: {resposta_teste.content}")

 LLM configurado
 Teste: Azul.


# Montar o RAG completo
### Aqui conectamos os dois sistemas:

##### Sistema 1 (busca): o banco vetorial como retriever
##### Sistema 2 (resposta): o LLM

####O RetrievalQA orquestra o fluxo completo:

##### 1.Recebe a pergunta
##### 2.Chama o retriever (busca no banco)
##### 3.Monta o prompt com os chunks encontrados
##### 4.Manda pro LLM
##### 5.Retorna a resposta

In [ ]:
# RAG Completo (LCEL — forma atual do LangChain)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Template do prompt
# {context} = os chunks encontrados no banco
# {question} = a pergunta do usuário
prompt = ChatPromptTemplate.from_template("""
Você é um assistente especializado nos documentos da empresa.
Responda a pergunta usando apenas o contexto abaixo.
Se não souber a resposta, diga que não encontrou nos documentos.

Contexto:
{context}

Pergunta: {question}
""")

# Função que formata os chunks em texto
def formatar_chunks(docs):
    textos = []

    for doc in docs:
        textos.append(doc.page_content)

    contexto = "\n\n".join(textos)

    return contexto

# Monta o RAG — conecta os dois sistemas
retriever = banco.as_retriever(search_kwargs={"k": 3})

rag = (
    {"context": retriever | formatar_chunks, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG montado com LCEL")
print("Sistema 1 (busca): retriever k=3")
print("Sistema 2 (resposta): Groq llama-3.3-70b temperature=0")

RAG montado com LCEL
Sistema 1 (busca): retriever k=3
Sistema 2 (resposta): Groq llama-3.3-70b temperature=0


# Fazendo perguntas

In [ ]:
# 1.Tokeniza a pergunta
# 2.Gera embedding da pergunta
# 3.Busca os 3 chunks mais próximos no banco
# 4.Monta o prompt: [chunks encontrados] + [pergunta]
# 5.Manda pro LLM
# 6.Retorna a resposta

pergunta = "Qual é a cláusula de reajuste?"
pergunta2 = "Qual é a empresa referenciada no contrato?"

resposta = rag.invoke(pergunta2)

print(f"Pergunta: {pergunta2}")
print()
print(f"Resposta:")
print(resposta)

Pergunta: Qual é a empresa referenciada no contrato?

Resposta:
A empresa referenciada no contrato é a Xperiun S.A.
